# The LLAMA of WallStreet: LLM data extraction and sentiment analysis 

**Read carefully the following instructions and suggestions before writing code.**

## Task description and suggestions

You work as Data Scientist in a company that develops algorithmic trading strategies. 
Your supervisor aims to develop a sentiment analysis pipeline for Reddit posts, with the objective of incorporating Reddit comments into an automated trading strategy.  
The goal is to leverage the content of these comments to extract actionable insights for identifying potential stock buying or selling opportunities. 

In particular, he wants a dataset with the `ticker` (or stock symbol: AAPL for Apple, TSLA for Tesla etc...) of the company and the general `sentiment`  towards that stock.

Moreover, he wants to be able to analyze comments every day using Leonardo and `SLURM`, **BUT**, unfortunately, he does not know how to write or execute code.
He only wants to send prompt in natural english.


The pipeline you are going to create does not have any particular constraint about the technologies to be used; however, here are **some important suggestions**:

1. The **number of comments to be processed is big**, we are talking about 100k comments. So, it is **strongly advised to parallelize your workflow in some way** (i.e. multitrheading or other options)... the choice is yours.

2. Alternatevely, you can analyze just a smaller subset of the dataframe, again, the choice is yours.

3. The data extraction task (i.e. verifying a comment is about companies and identifying a ticker associated to the comment) can be done leveraging "LLM knowledge". In the section "Data extraction example" you can find an example of how to use an LLM for data extraction.

4. Once you have identified the relevant comments and associated them with their respective stock tickers, the sentiment analysis task can be carried out using either a large language model (LLM) or a domain-specific pre-trained model (e.g. [cardiffnlp/twitter-roberta-base-sentiment-latest](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)) or any other sentiment analysis model. 
The choice is yours; however, keep in mind that if you plan to run the sentiment analysis on worker nodes, the model must be downloaded and made available on those nodes in advance.

5. Write down, in a Markdown cell, the architecture/plan/structure/technology you would use to convert this workflow into an agentic system that can send the `student_job_LLM.py` SLURM script to the Leonardo cluster and run the analysis above. Imagine that this would become a tool for data scientists in an organization: they would interact with chatbots, but sometimes ask for this analysis to be run on Leonardo and then receive the results. The idea is to reason about how this could become chatbot functionality implemented as an agent plus a certain number of tools. **HINT**: remember the example shown during Lesson 7 [`1_agent_applications_the_slurmjob_maker.ipynb`](../../lecture_8/notebooks/1_agent_applications_the_slurmjob_maker.ipynb).

6. Given the layout you described in point 5, the tools you defined, and the responsibilities of the agentic system, write the system prompt for this system.

7. **OPTIONAL**: actually write the agentic system. Implement it in this notebook, using the LLM we provide, and show that it prepares SLURM jobs that would launch `student_job_LLM.py` on Leonardo, get job status, and optionally fetch the results locally. **HINT**: remember the example shown during Lesson 7 [`1_agent_applications_the_slurmjob_maker.ipynb`](../../lecture_8/notebooks/1_agent_applications_the_slurmjob_maker.ipynb).


**Given the large number of comments, it may be helpful to use this notebook as a playground to test the pipeline on a small subset first. Once you've implemented and verified your logic here, you can move it to a .py script and run it as a non-interactive job. For a template, see the `student_job_LLM.py` and `config/LLM_start_job.job` scripts. That said, feel free to organize your code in whatever way works best for you.**

Note that in this notebook we limited the number of comments to be processed to 1000. Change the variable LIMIT if you want to make tests with more comments in this notebook.  

# Imports

In [ ]:
from datetime import datetime

from pydantic import BaseModel
from enum import Enum

from langchain_openai import ChatOpenAI

import pandas as pd

# General config

The following general configuration dir sets the LLM that you may use in your project.  

In [ ]:
t0 = datetime.now()
print(f"Execution started: {t0}")

INPUT_FILE = "reddit_comments.csv"
MODEL_NAME = "mistralai/Mistral-Small-3.2-24B-Instruct-2506"
MODEL_NAME = "google/gemma-4-31B-it"

VLLM_ENDPOINT = "http://127.0.0.1:8000/v1"
API_KEY = "password"
API_KEY = "bUon34Bu3o#2"

# OpenAI client pointing to our local model
llm = ChatOpenAI(base_url=VLLM_ENDPOINT, api_key=API_KEY, model=MODEL_NAME)

Execution started: 2026-05-28 16:09:00.049848


# Data extraction example

LLMs can be effectively used to solve data extraction tasks. For instance, consider a collection of legal documents where laws are cited in inconsistent formats. You can provide an LLM with a system prompt that includes examples of how laws are typically cited and instruct it to extract all legal citations from the text using a standardized response format. This approach enables the model to recognize and structure heterogeneous references in a consistent way.

Response format is specified as a Python class which extends a pydantic BaseModel class. This is all you need to know, don't worry you don't need to study pydantic...  
Below you can find an example.  

In [3]:
# Using structured outputes to constraint the answer to have a specific format

SYSTEM_PROMPT = """You are an helpful assistant trained to extract Laws citations from legal proceedings.
You will be given a sentence, your task is to extract citations using the following format: [{"article": "<article_no>", "code":"<law_collection>"}].

- <article_no> is a string containing the number of the cited article
- <law_collection> is the name of the code containing the cited law. Law collection can assume only 3 possible values: United States Code, Code of Federal Regulations, NA if the name of the document is not present or the input text does not contain any citation.

Here are some examples:

[INPUT]: Tomorrow I have an exam about USC art. 1
[OUTPUT]: [{"article": "1", "code": "United States Code"}]
[INPUT]: I like pizza!
[OUTPUT]: [{"article":"NA", "code": "NA"}]
[INPUT]: I found article 22 of the United States Code very clear. On the opposite, the art 13 of CFR is totally incomprensible.
[OUTPUT]: [{"article": "22", "code": "United States Code"}, {"article": "13", "code": "Code of Federal Regulations"}]
"""


class Codes(Enum):
    USC = "United States Code"
    CFR = "Code of Federal Regulations"
    NA = "NA"


class Citation(BaseModel):
    article: str
    code: Codes


class Citations(BaseModel):
    citations: list[Citation]


answ = llm.with_structured_output(Citations).invoke(
    input=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Did you study article 33 of the USC and article 3475-bis of USC?",
        },
    ],
    temperature=0,
)

answ

Citations(citations=[Citation(article='33', code=<Codes.USC: 'United States Code'>), Citation(article='3475-bis', code=<Codes.USC: 'United States Code'>)])

Then you can acces values by iterating over the elements of the list. The output of the llm invocation is a standard python object!

**If you get a connection error running the above cell, it's because the model is still being loaded. In this case you must wait a bit and try again...**

In [5]:
for citation in answ.citations:
    print(f"Article: {citation.article}\nCode: {citation.code.value}\n\n")

Article: 33
Code: United States Code


Article: 3475-bis
Code: United States Code




# Read data

In [18]:
import pandas as pd

df = pd.read_csv(INPUT_FILE)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101974 entries, 0 to 101973
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   datetime       101974 non-null  object
 1   subreddits     101974 non-null  object
 2   submission_id  101974 non-null  object
 3   comments       101974 non-null  object
dtypes: object(4)
memory usage: 3.1+ MB
None


,datetime,subreddits,submission_id,comments
0,2025-03-29 18:32:13,news,1jmrj8p,A British man has been praised for tackling a ...
1,2025-03-29 18:32:13,news,1jmrj8p,Glad they recognized him. He did a great thing.
2,2025-03-29 18:32:13,news,1jmrj8p,A news on a British tourist that doesn't invol...
3,2025-03-29 18:32:13,news,1jmrj8p,That guys needs some real love. Hopefully he A...
4,2025-03-29 18:32:13,news,1jmrj8p,"“I’m Millwall, mate”. Probably"


# Logical steps to follow

1. Extract one or more tickers from each comment. 

2. Associate a sentiment (very positive, positive, neutral, negative, very negative) to each comment. To perform this task you can use an LLM or a model trained specifically for this task;

3. Map sentiment to integers: 2, 1, 0, -1, -2; 

4. Once you have mapped sentiment to numbers, calculate a daily sentiment for each ticker; 

5. Calculate some metrics (e.g. min, max, average sentiment) for each ticker on a daily basis; 

6. Plot the sentiment trend for some tickers you find interesting (e.g. AAPL, META, etc.) 

# Test your pipeline here